# Limpieza de answers con OpenAI — dataset_300_curado
Extrae solo el bloque de comandos Cisco IOS valido usando OpenAI y devuelve NO_CODE cuando no haya codigo util o falte contexto en la pregunta.

**Requisitos:**
```bash
pip install openai pandas tqdm python-dotenv
```

## Configuracion

In [4]:
import os
import re
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path('.env'))
API_KEY = os.getenv('OPENAI_API_KEY')
MODEL = 'gpt-5-mini'
INPUT_PATH = 'dataset_300_curado.csv'
OUTPUT_PATH = 'dataset_300_cleaned.csv'
SAMPLE = 20  # usa un int para prueba rapida
REASONING_EFFORT = 'low'

if not API_KEY:
    raise SystemExit('No se encontro OPENAI_API_KEY en .env')

client = OpenAI(api_key=API_KEY)
print(f"API_KEY cargada: {'OK' if API_KEY else 'NO ENCONTRADA'}")
print(f'Modelo: {MODEL}')

API_KEY cargada: OK
Modelo: gpt-5-mini


## Prompt

In [5]:
SYSTEM_PROMPT = """You are a Cisco IOS configuration expert and dataset cleaner.
Your task is to extract the single most complete and general Cisco command block from a network configuration answer.

Rules:
- Return ONLY the Cisco IOS commands, nothing else.
- Use the question as grounding context for the answer.
- If the answer contains multiple code blocks (e.g. a general template and a specific example), keep ONLY the most general and complete one. Do NOT merge them.
- If a command appears repeated (once as explanation, once as example), keep it only once.
- Do NOT invent, add, modify, or complete any command. Copy them exactly as they appear in the original.
- Remove all prose, explanations, notes, and markdown formatting (no ```, no Router(config)# labels if they are not in the original).
- If the answer depends on context not explicitly present in the question (missing topology, interface names, IPs, policies, assumptions, or prior conversation), return exactly: NO_CODE
- If there is no valid Cisco command block, return exactly: NO_CODE

Output format: raw commands only, one per line, or NO_CODE exactly."""

## Cargar dataset

In [6]:
df = pd.read_csv(INPUT_PATH)
if SAMPLE:
    df = df.sample(n=SAMPLE, random_state=42).reset_index(drop=True)
    print(f'Modo prueba: {SAMPLE} filas')

print(f'Dataset cargado: {len(df)} filas')
display(df[['id', 'question', 'category']].head(2) if 'category' in df.columns else df[['id', 'question']].head(2))

Modo prueba: 20 filas
Dataset cargado: 20 filas


,id,question,category
0,4418,How do you configure per-flow admission for a ...,QOS
1,3600,What configuration command is required to enab...,CONNECTIVITY


## Funcion de limpieza

In [7]:
def normalize_model_output(raw: str) -> str:
    text = (raw or '').strip()
    if not text:
        return 'NO_CODE'

    fenced = re.search(r'```(?:[a-zA-Z]+)?\s*(.*?)\s*```', text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    text = text.strip('`').strip()
    if not text or text.upper() == 'NO_CODE':
        return 'NO_CODE'

    return text


def extract_clean_answer(model: str, question: str, answer: str) -> str:
    user_prompt = f'Question:\n{question}\n\nAnswer:\n{answer}'
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_prompt},
            ],
            max_completion_tokens=512,
            reasoning_effort=REASONING_EFFORT,
        )

        raw = (response.choices[0].message.content or '').strip()
        return normalize_model_output(raw)

    except Exception as e:
        print(f'  [ERROR] {e}')
        return 'ERROR'

## Pipeline de limpieza

In [8]:
cleaned = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Limpiando answers (OpenAI)'):
    clean = extract_clean_answer(MODEL, str(row['question']), str(row['answer']))
    cleaned.append(clean)

df['answer_clean'] = cleaned
print('Listo.')

Limpiando answers (OpenAI):   0%|          | 0/20 [00:00<?, ?it/s]

Listo.


## Guardar resultado

In [10]:
no_code = int((df['answer_clean'] == 'NO_CODE').sum())
error = int((df['answer_clean'] == 'ERROR').sum())
ok = len(df) - no_code - error

print(f'OK       : {ok}')
print(f'NO_CODE  : {no_code}')
print(f'ERROR    : {error}')

df.to_csv(OUTPUT_PATH, index=False)
print(f'\nGuardado en: {OUTPUT_PATH}')
df[['id', 'question', 'answer_clean']].head(20)

OK       : 18
NO_CODE  : 2
ERROR    : 0

Guardado en: dataset_300_cleaned.csv


,id,question,answer_clean
0,4418,How do you configure per-flow admission for a ...,policy-map <policy_map_name>\n class <class_n...
1,3600,What configuration command is required to enab...,configure terminal\nint port-channel 1\nload-b...
2,1512,How can we configure data collection to repeat...,Router(config)# snmp-server enable traps\nRout...
3,1597,How can I enable PPP authentication using Kerb...,aaa authentication ppp krb5\nexit\nshow runnin...
4,4481,What is the average shape configured for the v...,policy-map output_parent\n class class-defaul...
5,3452,How to implement QoS using policy-maps and cla...,Router# configure terminal\nRouter(config)# po...
6,4088,What is the default value for NBAR coarse-grai...,Router(config)# ip nbar classification granula...
7,1324,To ensure successful reception of multicast tr...,ip mroute 10.1.1.0 255.255.255.0 tunnel 0
8,2629,How can you enable privileged EXEC mode and cr...,NO_CODE
9,39,How do you configure the scheduling parameters...,config t\nip sla schedule 1 life 300 start-tim...
